# N151 · 多结构组合与时间序服务

**目标：** 从API契约倒推存储结构与查询计划。

**先修：** N042, N059, N019。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 时间键值；日志追加；K路归并；关注关系；有序时间；一致性约束。

## 从问题到算法

时间键值把每个key的修改日志保持有序，get查询时间前驱。消息流不应每次拼接全部历史并全量排序：每个关注用户的日志已经有序，可以从各自末项做K路归并，只取前10条。关注关系与消息日志分开存储，取消关注不删除历史消息。

## 最小实现

**本章接口：** `TimeMap.set/get`、`Twitter.postTweet/getNewsFeed/follow/unfollow`

In [1]:
from bisect import bisect_right
from heapq import heapify,heappop,heappush

class TimeMap:
    def __init__(self): self.history={}
    def set(self,key,value,timestamp):
        history=self.history.setdefault(key,[])
        if history and timestamp<=history[-1][0]: raise ValueError('timestamps per key must strictly increase')
        history.append((timestamp,value))
    def get(self,key,timestamp):
        history=self.history.get(key,[]); index=bisect_right(history,timestamp,key=lambda item:item[0])-1
        return history[index][1] if index>=0 else ''

class Twitter:
    def __init__(self): self.clock=0; self.posts={}; self.following={}
    def postTweet(self,userId,tweetId):
        self.clock+=1; self.posts.setdefault(userId,[]).append((self.clock,tweetId))
    def follow(self,followerId,followeeId):
        if followerId!=followeeId: self.following.setdefault(followerId,set()).add(followeeId)
    def unfollow(self,followerId,followeeId):
        self.following.get(followerId,set()).discard(followeeId)
    def getNewsFeed(self,userId):
        users=self.following.get(userId,set())|{userId}; heap=[]
        for user in users:
            posts=self.posts.get(user,[])
            if posts:
                time,tweet=posts[-1]; heap.append((-time,user,len(posts)-1,tweet))
        heapify(heap); answer=[]
        while heap and len(answer)<10:
            negative,user,index,tweet=heappop(heap); answer.append(tweet)
            if index:
                time,tweet=self.posts[user][index-1]; heappush(heap,(-time,user,index-1,tweet))
        return answer

## 正确性、前提与复杂度

每个日志严格按时间增加，二分即可查找最后一次不晚于查询的修改。K路堆中保存每个来源尚未取出的最新消息，弹出的就是所有来源中的最新一条，再用该来源前一条补位。全局单调clock消除并列时间，不依赖真实机器时钟精度。

**代价：** TimeMap追加均摊O(1)、查询O(log 每key记录数)；消息流初始化O(F)、输出最多10项O(10 log F)，F含自己及关注人数；日志空间O(消息数+关注边数)。接口按顺序调用，不包含分布式时钟或并发一致性保证。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

service=Twitter(); service.postTweet(1,101); service.postTweet(2,201); service.follow(1,2); first=service.getNewsFeed(1); service.unfollow(1,2)
show_table(['状态','用户1消息流'],[('关注2时',first),('取消关注2后',service.getNewsFeed(1))])
times=TimeMap(); times.set('x','old',3); times.set('x','new',7)
show_table(['查询时间','返回值'],[(t,times.get('x',t)) for t in [0,3,5,7,20]])

状态,用户1消息流
关注2时,"[201, 101]"
取消关注2后,[101]


查询时间,返回值
0,
3,old
5,old
7,new
20,new


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
times=TimeMap(); times.set('foo','bar',1); times.set('foo','bar2',4)
assert times.get('foo',0)=='' and times.get('foo',3)=='bar' and times.get('foo',4)=='bar2' and times.get('missing',9)==''
from random import Random
rng=Random(151); service=Twitter(); posts=[]; following={}; counter=0
for _ in range(600):
    op=rng.randrange(4); user=rng.randrange(7); other=rng.randrange(7)
    if op==0:
        counter+=1; service.postTweet(user,counter); posts.append((counter,user,counter))
    elif op==1:
        service.follow(user,other)
        if user!=other: following.setdefault(user,set()).add(other)
    elif op==2:
        service.unfollow(user,other); following.get(user,set()).discard(other)
    else:
        allowed=following.get(user,set())|{user}; expected=[tweet for time,author,tweet in reversed(posts) if author in allowed][:10]
        assert service.getNewsFeed(user)==expected
for user in range(7):
    service.follow(user,user); service.unfollow(user,user)
    allowed=following.get(user,set())|{user}
    assert service.getNewsFeed(user)==[tweet for time,author,tweet in reversed(posts) if author in allowed][:10]
print('N151: 所有本章断言通过')

N151: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 解释全量扫描慢模型的正确性与局限。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 限定题目要求而不扩展分布式系统。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 981. Time Based Key-Value Store（canonical）
- 355. Design Twitter（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。